# Lezione 13 · Casi d'uso reali e chiusura del corso

**Corso:** Advanced Deep Learning (apprendimento profondo avanzato)

Benvenuto all'ultima lezione del corso. Oggi non impari una tecnica nuova: ripassi in codice i componenti che già padroneggi e li colleghi a sistemi reali di intelligenza artificiale (IA) che hanno cambiato il loro campo, così come li presenta il capitolo 14 del manuale, *Casi d'uso reali*.

## Obiettivi della sessione di chiusura

Al termine di questa lezione sei in grado di:

- Ripassare, a colpo d'occhio, le 13 lezioni del corso e la tecnica centrale di ciascuna
- Caricare un modello di visione preaddestrato e usarlo per classificare un'immagine reale, come fanno i sistemi di visione artificiale in produzione
- Rivisitare la rete generativa avversaria (GAN) della Lezione 9 e rivedere il generatore e il discriminatore in competizione
- Rivisitare l'attenzione scalata per prodotto scalare della Lezione 8 e osservarne la mappa di calore su un esempio nuovo
- Individuare i prossimi passi concreti per continuare a formarti autonomamente sul deep learning

## Ripasso lampo del corso

Prima di entrare nel codice, ripassa a colpo d'occhio che cosa hai imparato in ciascuna delle 13 lezioni:

- **Lezione 1 — Introduzione all'IA avanzata e la tua prima rete neurale.** Che cosa rende avanzato il deep learning e il tuo primo perceptrone multistrato con retropropagazione (backpropagation)
- **Lezione 2 — Visione artificiale I: reti convoluzionali.** Filtri, stride, padding e pooling in una rete neurale convoluzionale (CNN)
- **Lezione 3 — Visione artificiale II: architetture avanzate.** Blocchi residui, degradazione nelle reti profonde e il tuo primo modello preaddestrato con trasferimento di apprendimento
- **Lezione 4 — Ottimizzazione I.** Discesa del gradiente, tasso di apprendimento e convergenza
- **Lezione 5 — Ottimizzazione II.** Momento, metodi del secondo ordine, Adam e regolarizzazione
- **Lezione 6 — Sequenze e reti ricorrenti.** Reti neurali ricorrenti (RNN), il problema del gradiente che svanisce o esplode, e come la memoria a lungo e breve termine (LSTM) lo risolve con le porte
- **Lezione 7 — Deep learning per il linguaggio naturale.** Tokenizzazione, embedding e un classificatore di testo, la base dell'elaborazione del linguaggio naturale (NLP)
- **Lezione 8 — Attenzione e Transformer.** Query, chiave e valore, e l'attenzione scalata per prodotto scalare implementata da zero
- **Lezione 9 — Reti generative avversarie.** Il generatore e il discriminatore di una GAN in competizione in un gioco minimax
- **Lezione 10 — Apprendimento non supervisionato.** Autoencoder e rappresentazioni apprese senza etichette
- **Lezione 11 — Modelli generativi moderni.** L'autoencoder variazionale (VAE), il trucco della riparametrizzazione e lo spazio latente
- **Lezione 12 — IA responsabile.** Bias, equità, interpretabilità e governance
- **Lezione 13 — Casi d'uso reali e chiusura.** Oggi: colleghi tutto ciò che hai visto a sistemi reali e decidi i tuoi prossimi passi

## Visione: un modello preaddestrato in azione

Il capitolo 14 del manuale collega la visione artificiale a casi come AlphaFold, che combina blocchi residui e attenzione per prevedere la forma di una proteina, e a qualsiasi sistema industriale che classifica immagini in produzione. Hai già costruito una rete neurale convoluzionale da zero nella Lezione 2 e hai usato un modello preaddestrato nella Lezione 3. Ripeti ora lo stesso schema in pochi secondi: carichi una ResNet-18 addestrata su ImageNet e la usi per classificare una nuova immagine, senza addestrare nulla.

In [ ]:
import urllib.request

import torch
from PIL import Image
from torchvision import models, transforms

# Carga la ResNet-18 con los pesos entrenados sobre ImageNet
modelo_vision = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
modelo_vision.eval()  # desactiva el dropout y fija las estadísticas de BatchNorm

# Descarga una imagen de ejemplo y la lista de las 1000 clases de ImageNet
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg",
    "imagen_ejemplo.jpg",
)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt",
    "clases_imagenet.txt",
)
with open("clases_imagenet.txt") as archivo:
    categorias = [linea.strip() for linea in archivo.readlines()]

imagen = Image.open("imagen_ejemplo.jpg").convert("RGB")

preprocesar = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
])
entrada = preprocesar(imagen).unsqueeze(0)

with torch.no_grad():
    salida = modelo_vision(entrada)
probabilidades = torch.nn.functional.softmax(salida[0], dim=0)

top5_prob, top5_idx = torch.topk(probabilidades, 5)

print("Predicciones de la ResNet-18 preentrenada:\n")
for probabilidad, indice in zip(top5_prob, top5_idx):
    print(f"{categorias[indice]:25s} {probabilidad.item() * 100:5.1f} %")

imagen

## Generazione: la tua GAN della Lezione 9 rivisitata

Il capitolo 14 collega la generazione di dati sintetici a StyleGAN, che genera volti che non esistono, e all'uso delle GAN per ampliare dataset medici scarsi senza compromettere la privacy di nessun paziente. Ripeti ora, in pochi passaggi, lo stesso esperimento della Lezione 9: un generatore e un discriminatore che competono finché i campioni generati si avvicinano alla distribuzione reale.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# --- Datos reales: mezcla de ocho gaussianas en círculo, igual que en la Clase 9 ---
def sample_real_data(n_samples, n_modes=8, radius=4.0, std=0.15):
    angles = 2 * np.pi * np.random.randint(0, n_modes, size=n_samples) / n_modes
    centers = np.stack([radius * np.cos(angles), radius * np.sin(angles)], axis=1)
    points = centers + std * np.random.randn(n_samples, 2)
    return torch.tensor(points, dtype=torch.float32)


LATENT_DIM = 2


# --- Generador: ruido latente -> punto en dos dimensiones ---
class Generator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, z):
        return self.net(z)


# --- Discriminador: punto en dos dimensiones -> logit de "realidad" ---
class Discriminator(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x)


generator = Generator().to(device)
discriminator = Discriminator().to(device)
criterion = nn.BCEWithLogitsLoss()
opt_g = torch.optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))

BATCH_SIZE = 256
N_STEPS = 1500  # entrenamiento corto: hoy repasas el patrón, no repites el entrenamiento completo de la Clase 9

for step in range(1, N_STEPS + 1):
    # 1) Actualiza el discriminador
    real_data = sample_real_data(BATCH_SIZE).to(device)
    z = torch.randn(BATCH_SIZE, LATENT_DIM, device=device)
    fake_data = generator(z).detach()

    real_labels = torch.ones(BATCH_SIZE, 1, device=device)
    fake_labels = torch.zeros(BATCH_SIZE, 1, device=device)

    loss_d = criterion(discriminator(real_data), real_labels) + criterion(discriminator(fake_data), fake_labels)
    opt_d.zero_grad()
    loss_d.backward()
    opt_d.step()

    # 2) Actualiza el generador con la pérdida no saturante
    z = torch.randn(BATCH_SIZE, LATENT_DIM, device=device)
    fake_data = generator(z)
    loss_g = criterion(discriminator(fake_data), real_labels)
    opt_g.zero_grad()
    loss_g.backward()
    opt_g.step()

    if step % 500 == 0:
        print(f"Paso {step} de {N_STEPS} | pérdida D: {loss_d.item():.3f} | pérdida G: {loss_g.item():.3f}")

# Muestra final: datos reales frente a datos generados tras el entrenamiento corto
with torch.no_grad():
    real_vis = sample_real_data(500).numpy()
    fake_vis = generator(torch.randn(500, LATENT_DIM, device=device)).cpu().numpy()

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(real_vis[:, 0], real_vis[:, 1], s=10, alpha=0.5, label="Datos reales")
ax.scatter(fake_vis[:, 0], fake_vis[:, 1], s=10, alpha=0.5, label="Datos generados")
ax.set_title(f"GAN revisitada - tras {N_STEPS} pasos de entrenamiento")
ax.legend(loc="upper right")
ax.set_xlim(-6, 6)
ax.set_ylim(-6, 6)
ax.set_aspect("equal")
plt.show()

## Attenzione: guardare dove conta

Il capitolo 14 collega l'attenzione alla traduzione automatica moderna e ad AlphaFold, che la usa per mettere in relazione amminoacidi lontani nella sequenza ma vicini nella struttura tridimensionale finale. Ripeti ora il calcolo della Lezione 8 — query, chiave e valore — su una frase nuova, e osserva di nuovo la mappa di calore dei pesi di attenzione.

In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(1)

# Frase nueva, distinta a la de la Clase 8, ya tokenizada en palabras
tokens = ["la", "traduccion", "automatica", "conecta", "idiomas", "distintos"]
n = len(tokens)
d = 8      # dimensión del embedding, pequeña solo para que el ejemplo sea legible
d_k = 8    # dimensión de consultas y claves

X = torch.randn(n, d)
W_q = torch.randn(d, d_k) * 0.5
W_k = torch.randn(d, d_k) * 0.5
W_v = torch.randn(d, d_k) * 0.5

Q = X @ W_q   # consultas: qué busca cada palabra
K = X @ W_k   # claves: qué ofrece cada palabra al ser comparada
V = X @ W_v   # valores: qué contenido entrega cada palabra si es elegida


def softmax_manual(z, dim=-1):
    z = z - z.max(dim=dim, keepdim=True).values
    exp_z = torch.exp(z)
    return exp_z / exp_z.sum(dim=dim, keepdim=True)


def atencion(Q, K, V):
    d_k = Q.shape[-1]
    puntuaciones = Q @ K.transpose(-2, -1)          # producto punto: consulta contra cada clave
    puntuaciones = puntuaciones / (d_k ** 0.5)       # escalado por raíz de d_k
    pesos = softmax_manual(puntuaciones, dim=-1)     # cada fila suma uno: es una distribución
    salida = pesos @ V                               # combinación ponderada de los valores
    return salida, pesos


salida, pesos_atencion = atencion(Q, K, V)

plt.figure(figsize=(6, 5))
plt.imshow(pesos_atencion.detach().numpy(), cmap="viridis")
plt.xticks(range(n), tokens, rotation=45, ha="right")
plt.yticks(range(n), tokens)
plt.xlabel("Clave (palabra observada)")
plt.ylabel("Consulta (palabra que pregunta)")
plt.title("Atención escalada por producto punto, revisitada")
plt.colorbar(label="Peso de atención")
plt.tight_layout()
plt.show()

## E adesso?

Terminare questo corso non è la fine del percorso, ma il punto di partenza. Ecco alcuni modi concreti per continuare a imparare:

- **Leggi gli articoli originali.** Inizia da *Attention Is All You Need* (il Transformer), *Generative Adversarial Networks* (Goodfellow e collaboratori), *Deep Residual Learning for Image Recognition* (ResNet) e il rapporto tecnico di AlphaFold pubblicato su Nature. Non serve capire ogni equazione alla prima lettura: leggi l'introduzione e le figure, e torna alla matematica quando ne hai bisogno
- **Unisciti a una comunità.** Papers With Code collega ogni articolo alla sua implementazione, Hugging Face raccoglie modelli, dataset e una comunità attiva, e i forum e i gruppi locali di IA ti mettono in contatto con persone che risolvono problemi simili ai tuoi
- **Costruisci un progetto tuo.** Scegli un problema reale del tuo contesto — agroalimentare, forestale, sanitario, biotecnologico o qualsiasi altro che conosci da vicino — e applicagli una delle tecniche di questo corso. Nulla insegna quanto fare il debug del proprio codice alle 11 di sera
- **Torna al capitolo 13 quando distribuisci qualcosa di reale.** Prima di mettere in produzione un modello qualsiasi, verifica il bias, la robustezza di fronte a input inattesi e l'impatto sulle persone coinvolte

**Continua a leggere articoli, a sperimentare con il codice e a chiederti non solo come funziona un modello, ma per che cosa e per chi lo stai costruendo.**

## Riepilogo dei punti chiave dell'intero corso

- Hai padroneggiato i fondamenti: neuroni, livelli, retropropagazione e discesa del gradiente, con e senza momento
- Hai costruito reti neurali convoluzionali e capito perché le connessioni residue permettono di addestrare reti molto profonde
- Hai modellato sequenze con reti neurali ricorrenti e memoria a lungo e breve termine, e hai superato il loro collo di bottiglia con l'attenzione
- Hai implementato da zero l'attenzione scalata per prodotto scalare, il componente centrale del Transformer
- Hai addestrato una rete generativa avversaria e un autoencoder variazionale, due modi diversi di generare dati nuovi
- Hai praticato l'apprendimento non supervisionato con autoencoder e rappresentazioni apprese senza etichette
- Hai valutato bias, equità e interpretabilità, e imparato a distribuire modelli in modo responsabile
- Hai visto oggi come questi stessi componenti — convoluzioni, attenzione, generazione avversaria — si combinano in sistemi reali come AlphaGo, AlphaFold, il traduttore neurale di Google, WaveNet, StyleGAN e i grandi modelli linguistici moderni

**Apri ora il notebook di pratica di accompagnamento, `../notebooks_practica/clase13_practica_casos_de_uso_cierre.ipynb`, e chiudi il corso mettendo in pratica tutto ciò che hai imparato.**